In [1]:
import os
import torch
import torch.nn as nn
import torchaudio
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel, AutoFeatureExtractor
from sklearn.metrics import roc_curve
from tqdm import tqdm


c:\Users\22K-4699\miniconda3\envs\deepfake_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.fftpack as fft
from tqdm import tqdm
from scipy.signal import lfilter

# --- Configuration (UPDATE THESE PATHS) ---
DATASET_ROOT = r'la_dataset\LA'

# Protocol files for each split
PROTOCOL_FILES = {
    'train': os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.train.trn.txt'),
    'dev': os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.dev.trl.txt')
}

# Output files
OUTPUT_FILES = {
    'train': 'lfcc_features_LA_TRAIN.npz',
    'dev': 'lfcc_features_LA_DEV.npz'
}

# LFCC Parameters (as per specification)
SAMPLE_RATE = 16000  # ASVspoof audio is 16kHz
FRAME_LENGTH_MS = 25  # 25 ms
FRAME_SHIFT_MS = 10   # 10 ms
FFT_SIZE = 512
NUM_FILTERS = 20      # Number of linear filters
NUM_LFCC = 20         # Number of LFCC coefficients
FIXED_FRAMES = 400    # Fixed length for CNN input (pad/truncate)

# Convert ms to samples
FRAME_LENGTH = int(FRAME_LENGTH_MS * SAMPLE_RATE / 1000)  # 400 samples
FRAME_SHIFT = int(FRAME_SHIFT_MS * SAMPLE_RATE / 1000)    # 160 samples


# --- 1. LFCC Feature Extraction Functions ---

def pre_emphasis(signal, coeff=0.97):
    """Apply pre-emphasis filter to amplify high frequencies."""
    return lfilter([1, -coeff], [1], signal)


def frame_signal(signal, frame_length, frame_shift):
    """
    Split signal into overlapping frames.
    Returns: (num_frames, frame_length)
    """
    signal_length = len(signal)
    num_frames = int(np.ceil((signal_length - frame_length) / frame_shift)) + 1
    
    # Pad signal if needed
    pad_length = (num_frames - 1) * frame_shift + frame_length
    padded_signal = np.pad(signal, (0, max(0, pad_length - signal_length)), mode='constant')
    
    # Create frames
    frames = np.zeros((num_frames, frame_length))
    for i in range(num_frames):
        start = i * frame_shift
        frames[i] = padded_signal[start:start + frame_length]
    
    return frames


def apply_hamming_window(frames):
    """Apply Hamming window to each frame."""
    hamming = np.hamming(frames.shape[1])
    return frames * hamming


def linear_filterbank(num_filters, fft_size, sample_rate):
    """
    Create linear filterbank (uniform spacing in Hz).
    Returns: (num_filters, fft_size//2 + 1)
    """
    low_freq = 0
    high_freq = sample_rate / 2
    
    # Linear spacing in Hz
    filter_points = np.linspace(low_freq, high_freq, num_filters + 2)
    
    # Convert Hz to FFT bin
    bins = np.floor((fft_size + 1) * filter_points / sample_rate).astype(int)
    
    filterbank = np.zeros((num_filters, fft_size // 2 + 1))
    
    for i in range(num_filters):
        left = bins[i]
        center = bins[i + 1]
        right = bins[i + 2]
        
        # Rising slope
        for j in range(left, center):
            filterbank[i, j] = (j - left) / (center - left)
        
        # Falling slope
        for j in range(center, right):
            filterbank[i, j] = (right - j) / (right - center)
    
    return filterbank


def extract_lfcc(audio_path, num_lfcc=NUM_LFCC, num_filters=NUM_FILTERS):
    """
    Extract LFCC features from audio file.
    Returns: (num_lfcc, num_frames) - ready for CNN
    """
    try:
        # Load audio
        signal, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        
        # Normalize
        signal = signal / (np.max(np.abs(signal)) + 1e-8)
        
        # Pre-emphasis
        signal = pre_emphasis(signal)
        
        # Frame the signal
        frames = frame_signal(signal, FRAME_LENGTH, FRAME_SHIFT)
        
        # Apply Hamming window
        frames = apply_hamming_window(frames)
        
        # Compute FFT
        mag_frames = np.abs(np.fft.rfft(frames, FFT_SIZE))
        power_frames = (mag_frames ** 2) / FFT_SIZE
        
        # Apply linear filterbank
        filterbanks = linear_filterbank(num_filters, FFT_SIZE, SAMPLE_RATE)
        filter_energies = np.dot(power_frames, filterbanks.T)
        filter_energies = np.where(filter_energies == 0, np.finfo(float).eps, filter_energies)
        
        # Take log
        log_filter_energies = np.log(filter_energies)
        
        # Apply DCT to get LFCC
        lfcc = fft.dct(log_filter_energies, axis=1, norm='ortho')[:, :num_lfcc]
        
        # Transpose to (num_lfcc, num_frames) for CNN
        lfcc = lfcc.T
        
        # Pad or truncate to fixed length
        num_frames = lfcc.shape[1]
        if num_frames < FIXED_FRAMES:
            # Pad with zeros
            pad_width = FIXED_FRAMES - num_frames
            lfcc = np.pad(lfcc, ((0, 0), (0, pad_width)), mode='constant')
        else:
            # Truncate
            lfcc = lfcc[:, :FIXED_FRAMES]
        
        # Add channel dimension: (1, num_lfcc, num_frames)
        lfcc = np.expand_dims(lfcc, axis=0)
        
        return lfcc
        
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None


# --- 2. Protocol Parsing Function ---

def parse_protocol(protocol_path, root_path):
    """
    Parse ASVspoof protocol file.
    Returns DataFrame with file_path and label.
    """
    data = []
    
    # Map split code to directory
    subdir_map = {
        'T': os.path.join(root_path, 'ASVspoof2019_LA_train', 'flac'),
        'D': os.path.join(root_path, 'ASVspoof2019_LA_dev', 'flac'),
        'E': os.path.join(root_path, 'ASVspoof2019_LA_eval', 'flac')
    }
    
    try:
        with open(protocol_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                
                # Format: SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY
                # Example: LA_0079 LA_T_1138215 - - bonafide
                
                file_id = parts[1]  # e.g., LA_T_1138215
                split_code = file_id.split('_')[1]  # Extract 'T', 'D', or 'E'
                
                # Label: bonafide = 0, spoof = 1
                label = 0 if parts[-1].lower() == 'bonafide' else 1
                
                # Build file path
                file_path = os.path.join(subdir_map[split_code], file_id + '.flac')
                
                data.append({
                    'file_path': file_path,
                    'label': label
                })
                
    except FileNotFoundError:
        print(f"Error: Protocol file not found at {protocol_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error parsing protocol: {e}")
        return pd.DataFrame()
    
    return pd.DataFrame(data)


# --- 3. Main Feature Extraction ---

def extract_features_for_split(split_name):
    """Extract LFCC features for a given split (train/dev/eval)."""
    
    print(f"\n{'='*60}")
    print(f"Processing {split_name.upper()} split")
    print(f"{'='*60}")
    
    protocol_file = PROTOCOL_FILES[split_name]
    output_file = OUTPUT_FILES[split_name]
    
    # Parse protocol
    df_protocol = parse_protocol(protocol_file, DATASET_ROOT)
    
    if df_protocol.empty:
        print(f"No data found for {split_name} split. Skipping...")
        return
    
    print(f"Total files in protocol: {len(df_protocol)}")
    print(f"Bonafide samples: {(df_protocol['label'] == 0).sum()}")
    print(f"Spoof samples: {(df_protocol['label'] == 1).sum()}")
    
    X_features = []
    y_labels = []
    skipped = 0
    
    # Extract features
    for idx, row in tqdm(df_protocol.iterrows(), 
                         total=len(df_protocol), 
                         desc=f"Extracting LFCC ({split_name})"):
        
        feature = extract_lfcc(row['file_path'])
        
        if feature is not None:
            X_features.append(feature)
            y_labels.append(row['label'])
        else:
            skipped += 1
    
    # Convert to numpy arrays
    X_features = np.array(X_features)  # Shape: (num_samples, 1, num_lfcc, num_frames)
    y_labels = np.array(y_labels)      # Shape: (num_samples,)
    
    # Save features
    np.savez_compressed(output_file, X=X_features, y=y_labels)
    
    print(f"\n--- {split_name.upper()} Feature Extraction Complete ---")
    print(f"Total features extracted: {len(X_features)}")
    print(f"Skipped files: {skipped}")
    print(f"Feature shape: {X_features.shape}")
    print(f"Label shape: {y_labels.shape}")
    print(f"Saved to: {output_file}")


# --- 4. Run Feature Extraction for All Splits ---

if __name__ == "__main__":
    print("="*60)
    print("ASVspoof 2019 LA - LFCC Feature Extraction")
    print("="*60)
    print(f"\nLFCC Configuration:")
    print(f"  Sample Rate: {SAMPLE_RATE} Hz")
    print(f"  Frame Length: {FRAME_LENGTH_MS} ms ({FRAME_LENGTH} samples)")
    print(f"  Frame Shift: {FRAME_SHIFT_MS} ms ({FRAME_SHIFT} samples)")
    print(f"  FFT Size: {FFT_SIZE}")
    print(f"  Number of Filters: {NUM_FILTERS}")
    print(f"  Number of LFCC: {NUM_LFCC}")
    print(f"  Fixed Frame Length: {FIXED_FRAMES}")
    print(f"  Output Shape: (1, {NUM_LFCC}, {FIXED_FRAMES})")
    
    # Extract features for each split
    for split in ['train', 'dev']:
        extract_features_for_split(split)
    
    print("\n" + "="*60)
    print("ALL FEATURE EXTRACTION COMPLETE!")
    print("="*60)
    print("\nNext steps:")
    print("1. Use lfcc_features_LA_TRAIN.npz for training")
    print("2. Use lfcc_features_LA_DEV.npz for validation")

ASVspoof 2019 LA - LFCC Feature Extraction

LFCC Configuration:
  Sample Rate: 16000 Hz
  Frame Length: 25 ms (400 samples)
  Frame Shift: 10 ms (160 samples)
  FFT Size: 512
  Number of Filters: 20
  Number of LFCC: 20
  Fixed Frame Length: 400
  Output Shape: (1, 20, 400)

Processing TRAIN split
Total files in protocol: 25380
Bonafide samples: 2580
Spoof samples: 22800


Extracting LFCC (train): 100%|██████████| 25380/25380 [06:48<00:00, 62.17it/s] 



--- TRAIN Feature Extraction Complete ---
Total features extracted: 25380
Skipped files: 0
Feature shape: (25380, 1, 20, 400)
Label shape: (25380,)
Saved to: lfcc_features_LA_TRAIN.npz

Processing DEV split
Total files in protocol: 24844
Bonafide samples: 2548
Spoof samples: 22296


Extracting LFCC (dev): 100%|██████████| 24844/24844 [06:24<00:00, 64.57it/s]



--- DEV Feature Extraction Complete ---
Total features extracted: 24844
Skipped files: 0
Feature shape: (24844, 1, 20, 400)
Label shape: (24844,)
Saved to: lfcc_features_LA_DEV.npz

ALL FEATURE EXTRACTION COMPLETE!

Next steps:
1. Use lfcc_features_LA_TRAIN.npz for training
2. Use lfcc_features_LA_DEV.npz for validation


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- Configuration ---
TRAIN_FEATURES = 'lfcc_features_LA_TRAIN.npz'
DEV_FEATURES = 'lfcc_features_LA_DEV.npz'

# Training hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 10  # Stop if no improvement for 10 epochs

# Model save path
MODEL_SAVE_PATH = 'best_cnn_model.pth'

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# --- 1. Dataset Class ---
class ASVspoofDataset(Dataset):
    """Dataset class for loading pre-extracted LFCC features."""
    
    def __init__(self, npz_file):
        """
        Load features from .npz file.
        Expected format:
            X: (num_samples, 1, num_lfcc, num_frames)
            y: (num_samples,)
        """
        data = np.load(npz_file)
        self.features = data['X']  # Shape: (N, 1, 20, 400)
        self.labels = data['y']    # Shape: (N,)
        
        print(f"Loaded {npz_file}:")
        print(f"  Features shape: {self.features.shape}")
        print(f"  Labels shape: {self.labels.shape}")
        print(f"  Bonafide (0): {(self.labels == 0).sum()}")
        print(f"  Spoof (1): {(self.labels == 1).sum()}")
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        feature = torch.FloatTensor(self.features[idx])
        label = torch.FloatTensor([self.labels[idx]])
        return feature, label


# --- 2. CNN Model Architecture ---
class LFCC_CNN(nn.Module):
    """
    Simple CNN for fake audio detection.
    Input: (batch, 1, 20, 400) - (batch, channels, lfcc_coeffs, time_frames)
    Output: (batch, 1) - probability of being spoof
    """
    
    def __init__(self):
        super(LFCC_CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: (32, 10, 200)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: (64, 5, 100)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: (128, 2, 50)
        )
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)  # Output: (128, 1, 1)
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)  # Output raw logits (no Sigmoid)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x


# --- 3. EER Calculation Function ---
def calculate_eer(y_true, y_scores):
    """
    Calculate Equal Error Rate (EER).
    
    Args:
        y_true: Ground truth labels (0=bonafide, 1=spoof)
        y_scores: Prediction scores (higher = more likely spoof)
    
    Returns:
        eer: Equal Error Rate (%)
        threshold: Threshold at EER
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    
    # Find where FAR = FRR
    eer_threshold = thresholds[np.nanargmin(np.abs(fnr - fpr))]
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
    
    return eer * 100, eer_threshold


# --- 4. Training Function ---
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    
    for features, labels in tqdm(dataloader, desc="Training", leave=False):
        features = features.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Apply sigmoid to get probabilities for EER calculation
        probs = torch.sigmoid(outputs)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(probs.detach().cpu().numpy())
    
    avg_loss = running_loss / len(dataloader)
    all_labels = np.array(all_labels).flatten()
    all_preds = np.array(all_preds).flatten()
    
    # Calculate EER for training set
    train_eer, _ = calculate_eer(all_labels, all_preds)
    
    return avg_loss, train_eer


# --- 5. Validation Function ---
def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Validation", leave=False):
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            # Apply sigmoid to get probabilities for EER calculation
            probs = torch.sigmoid(outputs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(probs.cpu().numpy())
    
    avg_loss = running_loss / len(dataloader)
    all_labels = np.array(all_labels).flatten()
    all_preds = np.array(all_preds).flatten()
    
    # Calculate EER
    val_eer, threshold = calculate_eer(all_labels, all_preds)
    
    return avg_loss, val_eer, threshold


# --- 6. Main Training Loop ---
def main():
    print("="*60)
    print("ASVspoof 2019 - CNN Training")
    print("="*60)
    
    # Load datasets
    print("\nLoading datasets...")
    train_dataset = ASVspoofDataset(TRAIN_FEATURES)
    dev_dataset = ASVspoofDataset(DEV_FEATURES)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    
    dev_loader = DataLoader(
        dev_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    # Initialize model
    print("\nInitializing model...")
    model = LFCC_CNN().to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Calculate pos_weight for class imbalance (bonafide vs spoof)
    num_bonafide = (train_dataset.labels == 0).sum()
    num_spoof = (train_dataset.labels == 1).sum()
    pos_weight = torch.tensor([num_spoof / num_bonafide], device=device)
    
    print(f"\nClass distribution in TRAIN set:")
    print(f"  Bonafide: {num_bonafide}")
    print(f"  Spoof: {num_spoof}")
    print(f"  Imbalance ratio: 1:{num_spoof/num_bonafide:.1f}")
    print(f"  Using pos_weight: {pos_weight.item():.2f}")
    
    # Loss with class weighting and optimizer
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Training tracking
    best_val_eer = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    train_eers = []
    val_eers = []
    
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    print(f"Early stopping patience: {EARLY_STOP_PATIENCE} epochs")
    print("="*60)
    
    # Training loop
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        
        # Train
        train_loss, train_eer = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validate
        val_loss, val_eer, threshold = validate(
            model, dev_loader, criterion, device
        )
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_eers.append(train_eer)
        val_eers.append(val_eer)
        
        # Print results
        print(f"  Train Loss: {train_loss:.4f} | Train EER: {train_eer:.2f}%")
        print(f"  Val Loss:   {val_loss:.4f} | Val EER:   {val_eer:.2f}%")
        print(f"  Threshold at EER: {threshold:.4f}")
        
        # Save best model based on validation EER
        if val_eer < best_val_eer:
            best_val_eer = val_eer
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_eer': val_eer,
                'threshold': threshold,
            }, MODEL_SAVE_PATH)
            print(f"  ✓ Best model saved! (Val EER: {val_eer:.2f}%)")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")
        
        # Early stopping
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    # Final results
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Best Validation EER: {best_val_eer:.2f}%")
    print(f"Model saved to: {MODEL_SAVE_PATH}")
    
    # Plot training curves
    plot_training_curves(train_losses, val_losses, train_eers, val_eers)


# --- 7. Plotting Function ---
def plot_training_curves(train_losses, val_losses, train_eers, val_eers):
    """Plot training and validation curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    ax1.plot(train_losses, label='Train Loss', marker='o')
    ax1.plot(val_losses, label='Val Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)
    
    # EER plot
    ax2.plot(train_eers, label='Train EER', marker='o')
    ax2.plot(val_eers, label='Val EER', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('EER (%)')
    ax2.set_title('Training and Validation EER')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    print(f"\nTraining curves saved to: training_curves.png")
    plt.close()


if __name__ == "__main__":
    main()

Using device: cuda
ASVspoof 2019 - CNN Training

Loading datasets...
Loaded lfcc_features_LA_TRAIN.npz:
  Features shape: (25380, 1, 20, 400)
  Labels shape: (25380,)
  Bonafide (0): 2580
  Spoof (1): 22800
Loaded lfcc_features_LA_DEV.npz:
  Features shape: (24844, 1, 20, 400)
  Labels shape: (24844,)
  Bonafide (0): 2548
  Spoof (1): 22296

Initializing model...
Model parameters: 109,761

Class distribution in TRAIN set:
  Bonafide: 2580
  Spoof: 22800
  Imbalance ratio: 1:8.8
  Using pos_weight: 8.84

Starting training for 50 epochs...
Early stopping patience: 10 epochs

Epoch [1/50]


  Train Loss: 1.1403 | Train EER: 45.93%
  Val Loss:   0.4723 | Val EER:   19.19%
  Threshold at EER: 0.9841
  ✓ Best model saved! (Val EER: 19.19%)

Epoch [2/50]


  Train Loss: 0.4721 | Train EER: 25.35%
  Val Loss:   0.4114 | Val EER:   14.76%
  Threshold at EER: 0.9731
  ✓ Best model saved! (Val EER: 14.76%)

Epoch [3/50]


  Train Loss: 0.3968 | Train EER: 15.70%
  Val Loss:   0.3621 | Val EER:   8.59%
  Threshold at EER: 0.9429
  ✓ Best model saved! (Val EER: 8.59%)

Epoch [4/50]


  Train Loss: 0.3011 | Train EER: 9.11%
  Val Loss:   0.3985 | Val EER:   7.34%
  Threshold at EER: 0.9976
  ✓ Best model saved! (Val EER: 7.34%)

Epoch [5/50]


...

  Train Loss: 0.0258 | Train EER: 0.89%
  Val Loss:   0.1294 | Val EER:   3.69%
  Threshold at EER: 0.9980
  No improvement (6/10)

Epoch [30/50]


  Train Loss: 0.0184 | Train EER: 0.66%
  Val Loss:   0.1283 | Val EER:   2.94%
  Threshold at EER: 0.9970
  No improvement (7/10)

Epoch [31/50]


  Train Loss: 0.0254 | Train EER: 0.70%
  Val Loss:   0.2107 | Val EER:   2.24%
  Threshold at EER: 0.9083
  No improvement (8/10)

Epoch [32/50]


  Train Loss: 0.0255 | Train EER: 0.81%
  Val Loss:   0.2755 | Val EER:   2.51%
  Threshold at EER: 0.8790
  No improvement (9/10)

Epoch [33/50]


  Train Loss: 0.0409 | Train EER: 1.36%
  Val Loss:   0.1419 | Val EER:   2.51%
  Threshold at EER: 0.9834
  No improvement (10/10)

Early stopping triggered after 33 epochs

TRAINING COMPLETE!
Best Validation EER: 2.12%
Model saved to: best_cnn_model.pth

Training curves saved to: training_curves.png


FEATURE EXTRACTION FOR EVAL DATA TO VERIFY THE ERR ACCURACY

In [3]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.fftpack as fft
from tqdm import tqdm
from scipy.signal import lfilter

# --- Configuration (UPDATE THESE PATHS) ---
DATASET_ROOT = r'la_dataset\LA'

# Protocol files for each split
PROTOCOL_FILES = {
    'eval': os.path.join(DATASET_ROOT, 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019.LA.cm.eval.trl.txt')
}

# Output files
OUTPUT_FILES = {
    'eval': 'lfcc_features_LA_EVAL.npz'
}

# LFCC Parameters (as per specification)
SAMPLE_RATE = 16000  # ASVspoof audio is 16kHz
FRAME_LENGTH_MS = 25  # 25 ms
FRAME_SHIFT_MS = 10   # 10 ms
FFT_SIZE = 512
NUM_FILTERS = 20      # Number of linear filters
NUM_LFCC = 20         # Number of LFCC coefficients
FIXED_FRAMES = 400    # Fixed length for CNN input (pad/truncate)

# Convert ms to samples
FRAME_LENGTH = int(FRAME_LENGTH_MS * SAMPLE_RATE / 1000)  # 400 samples
FRAME_SHIFT = int(FRAME_SHIFT_MS * SAMPLE_RATE / 1000)    # 160 samples


# --- 1. LFCC Feature Extraction Functions ---

def pre_emphasis(signal, coeff=0.97):
    """Apply pre-emphasis filter to amplify high frequencies."""
    return lfilter([1, -coeff], [1], signal)


def frame_signal(signal, frame_length, frame_shift):
    """
    Split signal into overlapping frames.
    Returns: (num_frames, frame_length)
    """
    signal_length = len(signal)
    num_frames = int(np.ceil((signal_length - frame_length) / frame_shift)) + 1
    
    # Pad signal if needed
    pad_length = (num_frames - 1) * frame_shift + frame_length
    padded_signal = np.pad(signal, (0, max(0, pad_length - signal_length)), mode='constant')
    
    # Create frames
    frames = np.zeros((num_frames, frame_length))
    for i in range(num_frames):
        start = i * frame_shift
        frames[i] = padded_signal[start:start + frame_length]
    
    return frames


def apply_hamming_window(frames):
    """Apply Hamming window to each frame."""
    hamming = np.hamming(frames.shape[1])
    return frames * hamming


def linear_filterbank(num_filters, fft_size, sample_rate):
    """
    Create linear filterbank (uniform spacing in Hz).
    Returns: (num_filters, fft_size//2 + 1)
    """
    low_freq = 0
    high_freq = sample_rate / 2
    
    # Linear spacing in Hz
    filter_points = np.linspace(low_freq, high_freq, num_filters + 2)
    
    # Convert Hz to FFT bin
    bins = np.floor((fft_size + 1) * filter_points / sample_rate).astype(int)
    
    filterbank = np.zeros((num_filters, fft_size // 2 + 1))
    
    for i in range(num_filters):
        left = bins[i]
        center = bins[i + 1]
        right = bins[i + 2]
        
        # Rising slope
        for j in range(left, center):
            filterbank[i, j] = (j - left) / (center - left)
        
        # Falling slope
        for j in range(center, right):
            filterbank[i, j] = (right - j) / (right - center)
    
    return filterbank


def extract_lfcc(audio_path, num_lfcc=NUM_LFCC, num_filters=NUM_FILTERS):
    """
    Extract LFCC features from audio file.
    Returns: (num_lfcc, num_frames) - ready for CNN
    """
    try:
        # Load audio
        signal, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        
        # Normalize
        signal = signal / (np.max(np.abs(signal)) + 1e-8)
        
        # Pre-emphasis
        signal = pre_emphasis(signal)
        
        # Frame the signal
        frames = frame_signal(signal, FRAME_LENGTH, FRAME_SHIFT)
        
        # Apply Hamming window
        frames = apply_hamming_window(frames)
        
        # Compute FFT
        mag_frames = np.abs(np.fft.rfft(frames, FFT_SIZE))
        power_frames = (mag_frames ** 2) / FFT_SIZE
        
        # Apply linear filterbank
        filterbanks = linear_filterbank(num_filters, FFT_SIZE, SAMPLE_RATE)
        filter_energies = np.dot(power_frames, filterbanks.T)
        filter_energies = np.where(filter_energies == 0, np.finfo(float).eps, filter_energies)
        
        # Take log
        log_filter_energies = np.log(filter_energies)
        
        # Apply DCT to get LFCC
        lfcc = fft.dct(log_filter_energies, axis=1, norm='ortho')[:, :num_lfcc]
        
        # Transpose to (num_lfcc, num_frames) for CNN
        lfcc = lfcc.T
        
        # Pad or truncate to fixed length
        num_frames = lfcc.shape[1]
        if num_frames < FIXED_FRAMES:
            # Pad with zeros
            pad_width = FIXED_FRAMES - num_frames
            lfcc = np.pad(lfcc, ((0, 0), (0, pad_width)), mode='constant')
        else:
            # Truncate
            lfcc = lfcc[:, :FIXED_FRAMES]
        
        # Add channel dimension: (1, num_lfcc, num_frames)
        lfcc = np.expand_dims(lfcc, axis=0)
        
        return lfcc
        
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None


# --- 2. Protocol Parsing Function ---

def parse_protocol(protocol_path, root_path):
    """
    Parse ASVspoof protocol file.
    Returns DataFrame with file_path and label.
    """
    data = []
    
    # Map split code to directory
    subdir_map = {
        'T': os.path.join(root_path, 'ASVspoof2019_LA_train', 'flac'),
        'D': os.path.join(root_path, 'ASVspoof2019_LA_dev', 'flac'),
        'E': os.path.join(root_path, 'ASVspoof2019_LA_eval', 'flac')
    }
    
    try:
        with open(protocol_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                
                # Format: SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY
                # Example: LA_0079 LA_T_1138215 - - bonafide
                
                file_id = parts[1]  # e.g., LA_T_1138215
                split_code = file_id.split('_')[1]  # Extract 'T', 'D', or 'E'
                
                # Label: bonafide = 0, spoof = 1
                label = 0 if parts[-1].lower() == 'bonafide' else 1
                
                # Build file path
                file_path = os.path.join(subdir_map[split_code], file_id + '.flac')
                
                data.append({
                    'file_path': file_path,
                    'label': label
                })
                
    except FileNotFoundError:
        print(f"Error: Protocol file not found at {protocol_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error parsing protocol: {e}")
        return pd.DataFrame()
    
    return pd.DataFrame(data)


# --- 3. Main Feature Extraction ---

def extract_features_for_split(split_name):
    """Extract LFCC features for a given split (train/dev/eval)."""
    
    print(f"\n{'='*60}")
    print(f"Processing {split_name.upper()} split")
    print(f"{'='*60}")
    
    protocol_file = PROTOCOL_FILES[split_name]
    output_file = OUTPUT_FILES[split_name]
    
    # Parse protocol
    df_protocol = parse_protocol(protocol_file, DATASET_ROOT)
    
    if df_protocol.empty:
        print(f"No data found for {split_name} split. Skipping...")
        return
    
    print(f"Total files in protocol: {len(df_protocol)}")
    print(f"Bonafide samples: {(df_protocol['label'] == 0).sum()}")
    print(f"Spoof samples: {(df_protocol['label'] == 1).sum()}")
    
    X_features = []
    y_labels = []
    skipped = 0
    
    # Extract features
    for idx, row in tqdm(df_protocol.iterrows(), 
                         total=len(df_protocol), 
                         desc=f"Extracting LFCC ({split_name})"):
        
        feature = extract_lfcc(row['file_path'])
        
        if feature is not None:
            X_features.append(feature)
            y_labels.append(row['label'])
        else:
            skipped += 1
    
    # Convert to numpy arrays
    X_features = np.array(X_features)  # Shape: (num_samples, 1, num_lfcc, num_frames)
    y_labels = np.array(y_labels)      # Shape: (num_samples,)
    
    # Save features
    np.savez_compressed(output_file, X=X_features, y=y_labels)
    
    print(f"\n--- {split_name.upper()} Feature Extraction Complete ---")
    print(f"Total features extracted: {len(X_features)}")
    print(f"Skipped files: {skipped}")
    print(f"Feature shape: {X_features.shape}")
    print(f"Label shape: {y_labels.shape}")
    print(f"Saved to: {output_file}")


# --- 4. Run Feature Extraction for All Splits ---

if __name__ == "__main__":
    print("="*60)
    print("ASVspoof 2019 LA - LFCC Feature Extraction")
    print("="*60)
    print(f"\nLFCC Configuration:")
    print(f"  Sample Rate: {SAMPLE_RATE} Hz")
    print(f"  Frame Length: {FRAME_LENGTH_MS} ms ({FRAME_LENGTH} samples)")
    print(f"  Frame Shift: {FRAME_SHIFT_MS} ms ({FRAME_SHIFT} samples)")
    print(f"  FFT Size: {FFT_SIZE}")
    print(f"  Number of Filters: {NUM_FILTERS}")
    print(f"  Number of LFCC: {NUM_LFCC}")
    print(f"  Fixed Frame Length: {FIXED_FRAMES}")
    print(f"  Output Shape: (1, {NUM_LFCC}, {FIXED_FRAMES})")
    
    # Extract features for eval split only
    for split in ['eval']:
        extract_features_for_split(split)
    
    print("\n" + "="*60)
    print("EVAL FEATURE EXTRACTION COMPLETE!")
    print("="*60)
    print("\nNext steps:")
    print("1. Use lfcc_features_LA_EVAL.npz for final testing only")

ASVspoof 2019 LA - LFCC Feature Extraction

LFCC Configuration:
  Sample Rate: 16000 Hz
  Frame Length: 25 ms (400 samples)
  Frame Shift: 10 ms (160 samples)
  FFT Size: 512
  Number of Filters: 20
  Number of LFCC: 20
  Fixed Frame Length: 400
  Output Shape: (1, 20, 400)

Processing EVAL split
Total files in protocol: 71237
Bonafide samples: 7355
Spoof samples: 63882


Extracting LFCC (eval): 100%|██████████| 71237/71237 [48:52<00:00, 24.29it/s] 



--- EVAL Feature Extraction Complete ---
Total features extracted: 71237
Skipped files: 0
Feature shape: (71237, 1, 20, 400)
Label shape: (71237,)
Saved to: lfcc_features_LA_EVAL.npz

EVAL FEATURE EXTRACTION COMPLETE!

Next steps:
1. Use lfcc_features_LA_EVAL.npz for final testing only


In [5]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- Configuration ---
EVAL_FEATURES  = 'lfcc_features_LA_EVAL.npz'
MODEL_SAVE_PATH = 'best_cnn_model.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# --- 1. Dataset Class ---
class ASVspoofDataset(Dataset):
    def __init__(self, npz_file):
        data = np.load(npz_file)
        self.features = data['X']
        self.labels   = data['y']
        print(f"Loaded {npz_file}:")
        print(f"  Features shape : {self.features.shape}")
        print(f"  Bonafide (0)   : {(self.labels == 0).sum()}")
        print(f"  Spoof    (1)   : {(self.labels == 1).sum()}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature = torch.FloatTensor(self.features[idx])
        label   = torch.FloatTensor([self.labels[idx]])
        return feature, label


# --- 2. Same CNN Architecture (must match saved model) ---
class LFCC_CNN(nn.Module):
    def __init__(self):
        super(LFCC_CNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)   # raw logits
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


# --- 3. EER Calculation ---
def calculate_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer       = fpr[idx] * 100
    threshold = thresholds[idx]
    return eer, threshold


# --- 4. Evaluation on Eval Set ---
def evaluate(model, dataloader):
    model.eval()
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Evaluating"):
            features = features.to(device)
            outputs  = model(features)
            probs    = torch.sigmoid(outputs)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels).flatten()
    all_probs  = np.array(all_probs).flatten()
    return all_labels, all_probs


# --- 5. Plot ROC Curve ---
def plot_roc(y_true, y_scores, eer, threshold):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='steelblue', lw=2, label='ROC Curve')
    plt.scatter([eer/100], [1 - eer/100], color='red', zorder=5,
                label=f'EER = {eer:.2f}%  (threshold={threshold:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate (FAR)')
    plt.ylabel('True Positive Rate (1 - FRR)')
    plt.title('ROC Curve — Eval Set')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('eval_roc_curve.png', dpi=150)
    print("ROC curve saved to: eval_roc_curve.png")
    plt.close()


# --- 6. Main ---
def main():
    print("=" * 60)
    print("ASVspoof 2019 — Final Evaluation on EVAL Set")
    print("=" * 60)

    # Load eval data
    print("\nLoading eval dataset...")
    eval_dataset = ASVspoofDataset(EVAL_FEATURES)
    eval_loader  = DataLoader(eval_dataset, batch_size=64,
                              shuffle=False, num_workers=0)

    # Load saved model
    print("\nLoading saved model...")
    model = LFCC_CNN().to(device)
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"  Loaded model from epoch {checkpoint['epoch'] + 1}")
    print(f"  Dev EER when saved : {checkpoint['val_eer']:.2f}%")

    # Run evaluation
    print("\nRunning evaluation...")
    y_true, y_scores = evaluate(model, eval_loader)

    # Compute EER
    eer, threshold = calculate_eer(y_true, y_scores)

    # Results
    print("\n" + "=" * 60)
    print("FINAL RESULTS ON EVAL SET")
    print("=" * 60)
    print(f"  Eval EER  : {eer:.2f}%")
    print(f"  Threshold : {threshold:.4f}")
    print(f"  Dev EER   : {checkpoint['val_eer']:.2f}%  (for comparison)")
    print("=" * 60)

    # Plot ROC
    plot_roc(y_true, y_scores, eer, threshold)


if __name__ == "__main__":
    main()

Using device: cuda
ASVspoof 2019 — Final Evaluation on EVAL Set

Loading eval dataset...
Loaded lfcc_features_LA_EVAL.npz:
  Features shape : (71237, 1, 20, 400)
  Bonafide (0)   : 7355
  Spoof    (1)   : 63882

Loading saved model...
  Loaded model from epoch 23
  Dev EER when saved : 2.12%

Running evaluation...


Evaluating: 100%|██████████| 1114/1114 [00:05<00:00, 187.76it/s]



FINAL RESULTS ON EVAL SET
  Eval EER  : 6.39%
  Threshold : 0.4623
  Dev EER   : 2.12%  (for comparison)
ROC curve saved to: eval_roc_curve.png
